# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset elements (record sets, fields, columns) are referenced by their Croissant schema `@id`.

### Dataset Source
The dataset source is accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata (do not subscript metadata; access as attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Authors: {dataset.metadata.author}\n")
print(f"Cite as: {dataset.metadata.cite_as}\n")


## 2. Data Overview
Review available record sets and their fields/columns and their `@id`s.

In [ ]:
# List all record sets defined by their '@id'.
record_sets = dataset.metadata.record_sets

if not record_sets:
    print('No record sets found. The main data is likely stored as a single main record set.')

# In FAIR^2, the key record set is the main file, so fetch from dataset.metadata.distribution or files
# List distributions (usually direct data files containing record sets)
dists = dataset.metadata.distribution
for dist in dists:
    print('Data Distribution:')
    pprint.pprint(dist, indent=2)

print('\n--- Record Sets Overview ---')
# List all record sets and their associated fields by @id
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    print(f"  Description: {rs.get('description')}")
    print(f"  Fields:")
    for field in rs.get('field', []):
        print(f"    - Field @id: {field['@id']}  Name: {field.get('name')}")
    print()
if not dataset.metadata.record_sets:
    print('No explicit record sets defined in the schema. Proceeding to access records directly.')

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All field and record set references use their `@id`s, per Croissant conventions.

The main record set is typically identified as the primary table or dataset file. Here, we infer its `@id` from the available metadata.

In [ ]:
# Extract from the main record set.
# If record_sets is empty, mlcroissant supports loading records without specifying record_set; otherwise, use the explicit @id.

# List all record sets by @id for reference
main_record_set_id = None
if dataset.metadata.record_sets:
    main_record_set_id = dataset.metadata.record_sets[0]['@id']
    print(f"Using primary record set: {main_record_set_id}")
else:
    # Try using the only distribution as an implicit record set
    if dataset.metadata.distribution:
        main_record_set_id = dataset.metadata.distribution[0]['@id']
        print('No record sets defined, using distribution as the data source.')

# Load records from the chosen record set
try:
    records = list(dataset.records(record_set=main_record_set_id))
except Exception as e:
    print(f"Could not load records using record_set={main_record_set_id}:", e)
    records = list(dataset.records())

# Load into dataframe
df = pd.DataFrame(records)

print('Fields in main record set:')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records by a numeric field value, normalize a numeric column, and group by a categorical column. Remember, all field accesses use their Croissant `@id` from the schema (or equivalently the column header if not explicitly mapped).

Typical numeric fields for a protein dataset include e.g. `MW` (molecular weight), `Score`, or abundance measures.

In [ ]:
# Select a numeric field (by Croissant '@id' or column name as loaded in DataFrame)

numeric_fields = [col for col in df.columns if any(key in col.lower() for key in ['score', 'mw', 'abundance', 'count', 'coverage', 'peptide'])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Take first candidate
    print(f"Selected numeric field for analysis: {numeric_field_id}")
else:
    numeric_field_id = df.columns[0]  # fallback
    print(f"No obvious numeric field, using: {numeric_field_id}")

# Filter records: Keep records with the numeric field above its median value
threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

# Often, dtype is object for string inputs: Try to convert to float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (choose an id with string values, like 'description', 'protein', or 'accession')
group_field_candidates = [
    col for col in df.columns if col.lower() in ['accession', 'protein', 'description', 'gene', 'group']
]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (means):")
    print(grouped_df.head())
else:
    print("No valid group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships between numeric fields using matplotlib or seaborn. Adjust field references by their names as loaded from the Croissant schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (filtered)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field is available, show boxplot
    if group_field and group_field in filtered_df.columns:
        top_groups = filtered_df[group_field].value_counts().index[:5]
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df[filtered_df[group_field].isin(top_groups)])
        plt.title(f'{numeric_field_id} by {group_field} (top 5 groups)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using its Croissant schema with `mlcroissant`
- Examined available record sets, fields, and metadata (referenced always by their Croissant `@id`)
- Loaded the main record set to a DataFrame
- Explored and normalized a key numeric field, and grouped by a protein/categorical attribute where possible
- Visualized the data distributions using Matplotlib and Seaborn

This workflow provides a template for analyzing any croissant-standardized dataset using `mlcroissant` and pandas. For additional analyses, refer to field/column @id values in the dataset schema.